[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/37.%20The%20Equipment%20Selection%20Problem%20%28RTG%20vs.%20RMG%20vs.%20Straddle%29/P37-Tier-5_executed.ipynb)

---


# 37. The Equipment Selection Problem (RTG vs. RMG vs. Straddle)
## Tier 5: An Integrated Digital Twin Simulation

### Goal
Build a small, self-contained simulation that connects equipment data, simulated sensor readings, a virtual equipment state, heuristic forecasts, and what-if scenarios. The notebook illustrates how these components can support operational monitoring and analysis.

### Inputs and outputs
The inputs are the equipment-type specifications, a simulated fleet configuration, and scenario definitions. The notebook produces simulated sensor readings, equipment and system KPIs, heuristic forecasts, scenario metrics, and visualizations. NumPy and pandas-style numerical workflows are used with the Python libraries imported below; the equipment model, simulation logic, and reporting are implemented directly in Python.

### Scope and modeling assumptions
- Sensor readings and environmental conditions are simulated; the notebook does not connect to physical IoT devices or a live data stream.
- The prediction models are transparent heuristic generators, not trained forecasting models. Their accuracy values are illustrative assumptions and are not estimated from validation data.
- Scenario results are simplified what-if calculations, not operational recommendations validated against a terminal.
- The notebook demonstrates monitoring and scenario analysis. It does not formulate or solve an equipment-selection optimization problem, and it does not establish that the illustrated fleet is optimal.
- Productivity is measured in containers per hour. Equipment energy intensity is measured in kWh per container; multiplying intensity by throughput gives an energy-consumption rate in kWh per hour.

### Solution sequence
1. Define equipment specifications and data structures for sensors and twin states.
2. Initialize a simulated fleet and deploy simulated sensors.
3. Generate readings, aggregate them into a virtual system state, and calculate KPIs.
4. Generate heuristic forecasts from the latest state.
5. Evaluate predefined scenarios and compare them with normal operations.
6. Visualize and summarize the simulated results.

### Interpreting results
The outputs describe this simulation and its assumptions. A forecast is not evidence of predictive accuracy, and a scenario comparison is not a causal estimate of real-world performance. In an operational digital twin, sensor calibration, time alignment, model validation, uncertainty analysis, and safeguards for decisions would require separate data and engineering work.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, List, Optional
import random

import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)
random.seed(42)

print('Equipment Selection: Integrated Digital Twin Simulation')
print('Simulated monitoring, heuristic forecasting, and scenario analysis')

### Equipment and Digital Twin Data Structure

A data structure gives each concept a clear set of fields. Equipment specifications describe the equipment types; sensor records hold individual simulated readings; and a twin state groups readings and KPIs at one simulation time. These structures make the information passed between simulation steps explicit.

In [ ]:
@dataclass
class EquipmentType:
    name: str
    capital_cost: float  # Million USD
    operating_cost: float  # Million USD per year
    productivity: float  # Containers per hour at full utilization
    flexibility: float  # Scale from 0 to 1
    maintenance: float  # Million USD per year
    footprint: float  # Square meters
    energy_consumption: float  # kWh per container
    sensor_count: int
    reliability: float  # Scale from 0 to 1


@dataclass
class IoTSensor:
    sensor_id: str
    equipment_id: str
    sensor_type: str
    reading_value: float
    timestamp: datetime
    accuracy: float  # Scale from 0 to 1; used as a simulation factor


@dataclass
class DigitalTwinState:
    timestamp: datetime
    equipment_states: Dict[str, Dict]
    environmental_conditions: Dict[str, float]
    operational_kpis: Dict[str, float]
    prediction_confidence: float


equipment_types = {
    'RTG': EquipmentType(
        name='Rubber-Tired Gantry',
        capital_cost=2.5,
        operating_cost=0.8,
        productivity=25,
        flexibility=0.8,
        maintenance=0.3,
        footprint=200,
        energy_consumption=3.2,
        sensor_count=15,
        reliability=0.92,
    ),
    'RMG': EquipmentType(
        name='Rail-Mounted Gantry',
        capital_cost=3.2,
        operating_cost=0.6,
        productivity=35,
        flexibility=0.4,
        maintenance=0.25,
        footprint=150,
        energy_consumption=2.8,
        sensor_count=20,
        reliability=0.95,
    ),
    'Straddle': EquipmentType(
        name='Straddle Carrier',
        capital_cost=4.1,
        operating_cost=1.1,
        productivity=20,
        flexibility=0.9,
        maintenance=0.4,
        footprint=100,
        energy_consumption=4.5,
        sensor_count=12,
        reliability=0.88,
    ),
}

print('Equipment types and selected specifications:')
for equipment_code, equipment in equipment_types.items():
    print(f'{equipment_code}: {equipment.name}')
    print(f'  Capital cost: ${equipment.capital_cost:.1f} million')
    print(f'  Full-utilization productivity: {equipment.productivity} containers/hour')
    print(f'  Energy intensity: {equipment.energy_consumption} kWh/container')
    print(f'  Sensors per unit: {equipment.sensor_count}')
    print(f'  Nominal reliability: {equipment.reliability:.2f}')
    print()

### Digital Twin System Implementation

The system follows a data-flow sequence: simulated equipment generates sensor readings; readings are grouped by equipment; aggregated values form a timestamped twin state; and that state supplies inputs to the forecast and scenario steps. The class keeps these steps visible as separate methods so that each part of the simulation can be inspected.

The simulated sensor accuracy factor and random noise are modeling devices, not a calibrated measurement-error model. Likewise, the forecast formulas below are deliberately simple heuristics. Their purpose is to demonstrate how a prediction interface can use the latest state, not to claim validated forecasting performance.

In [ ]:
class IntegratedDigitalTwinSystem:
    def __init__(self, equipment_types: Dict[str, EquipmentType]):
        self.equipment_types = equipment_types
        self.equipment_units = {}
        self.iot_sensors = []
        self.digital_twin_states = []
        self.prediction_models = {}
        self.scenario_results = {}

        # These are illustrative system-level assumptions, not measured results.
        self.synchronization_accuracy = 0.95
        self.prediction_accuracy = 0.87
        self.system_uptime = 0.99

        self._initialize_equipment_fleet()
        self._deploy_iot_sensors()
        self._initialize_prediction_models()

    def _initialize_equipment_fleet(
        self, equipment_config: Optional[Dict[str, int]] = None
    ):
        """Create simulated units with equipment-specific performance variation."""
        if equipment_config is None:
            equipment_config = {'RTG': 12, 'RMG': 8, 'Straddle': 15}

        unit_id_counter = 1
        for equipment_code, number_of_units in equipment_config.items():
            if equipment_code not in self.equipment_types:
                raise ValueError(f'Unknown equipment type: {equipment_code}')
            if number_of_units < 0:
                raise ValueError('Equipment counts must be non-negative.')

            for _ in range(number_of_units):
                unit_id = f'{equipment_code}_{unit_id_counter:03d}'
                equipment = self.equipment_types[equipment_code]

                # Unit-level variation represents heterogeneity within a type.
                productivity = equipment.productivity * (
                    1 + np.random.normal(0, 0.1)
                )
                reliability = np.clip(
                    equipment.reliability * (1 + np.random.normal(0, 0.05)),
                    0.7,
                    1.0,
                )

                self.equipment_units[unit_id] = {
                    'equipment_type': equipment_code,
                    'productivity': productivity,
                    'reliability': reliability,
                    'utilization': 0.75,
                    'maintenance_hours': 0.0,
                    'status': 'ACTIVE',
                    'age_years': np.random.uniform(0, 10),
                    'operating_hours': np.random.uniform(1000, 10000),
                }
                unit_id_counter += 1

        print(f'Initialized {len(self.equipment_units)} equipment units:')
        for equipment_code in equipment_config:
            count = sum(
                unit['equipment_type'] == equipment_code
                for unit in self.equipment_units.values()
            )
            print(f'  {equipment_code}: {count} units')

    def _deploy_iot_sensors(self):
        """Create simulated sensor records for every unit in the fleet."""
        sensor_id_counter = 1
        sensor_types = [
            'productivity', 'energy', 'maintenance', 'environment', 'vibration'
        ]

        for unit_id, unit in self.equipment_units.items():
            equipment_code = unit['equipment_type']
            sensor_count = self.equipment_types[equipment_code].sensor_count

            for index in range(sensor_count):
                sensor = IoTSensor(
                    sensor_id=f'SENSOR_{sensor_id_counter:04d}',
                    equipment_id=unit_id,
                    sensor_type=sensor_types[index % len(sensor_types)],
                    reading_value=0.0,
                    timestamp=datetime.now(),
                    accuracy=np.random.uniform(0.85, 0.98),
                )
                self.iot_sensors.append(sensor)
                sensor_id_counter += 1

        print(f'Deployed {len(self.iot_sensors)} simulated sensors.')

    def _initialize_prediction_models(self):
        """Register heuristic forecast categories and illustrative metadata."""
        self.prediction_models = {
            'productivity': {
                'model_type': 'heuristic time-series rule',
                'accuracy': 0.87,
                'update_frequency': 'hourly',
            },
            'maintenance': {
                'model_type': 'heuristic alert projection',
                'accuracy': 0.82,
                'update_frequency': 'daily',
            },
            'energy': {
                'model_type': 'heuristic consumption projection',
                'accuracy': 0.91,
                'update_frequency': 'real-time',
            },
            'failure': {
                'model_type': 'heuristic risk projection',
                'accuracy': 0.79,
                'update_frequency': 'weekly',
            },
        }

        print('Registered illustrative forecast categories:')
        for model_name, model_info in self.prediction_models.items():
            print(
                f'  {model_name}: {model_info["model_type"]} '
                f'(assumed accuracy: {model_info["accuracy"]:.2f})'
            )

    def collect_sensor_data(self, current_time: datetime) -> List[IoTSensor]:
        """Generate one simulated reading for each sensor at the given time."""
        sensor_readings = []

        for sensor in self.iot_sensors:
            unit = self.equipment_units[sensor.equipment_id]
            base_reading = self._calculate_sensor_reading(
                sensor.sensor_type, unit, current_time
            )

            # Noise and the accuracy factor make the synthetic readings imperfect.
            noise = np.random.normal(0, 0.05)
            sensor.reading_value = base_reading * (1 + noise) * sensor.accuracy
            sensor.timestamp = current_time
            sensor_readings.append(sensor)

        return sensor_readings

    def _calculate_sensor_reading(
        self, sensor_type: str, unit: Dict, current_time: datetime
    ) -> float:
        """Compute a synthetic reading from a unit's simulated operating state."""
        equipment = self.equipment_types[unit['equipment_type']]
        hour_factor = 0.7 + 0.3 * np.sin(current_time.hour * np.pi / 12)
        weather_factor = np.random.uniform(0.9, 1.1)
        throughput = unit['productivity'] * unit['utilization']

        if sensor_type == 'productivity':
            return throughput * hour_factor * weather_factor
        if sensor_type == 'energy':
            # Energy intensity times throughput gives an hourly energy rate.
            return (
                equipment.energy_consumption
                * throughput
                * weather_factor
            )
        if sensor_type == 'maintenance':
            return unit['maintenance_hours'] / 100
        if sensor_type == 'environment':
            return np.random.uniform(0.8, 1.2)
        if sensor_type == 'vibration':
            return (1 - unit['reliability']) * 10
        return 0.0

    def update_digital_twin_state(
        self, sensor_data: List[IoTSensor], current_time: datetime
    ) -> DigitalTwinState:
        """Aggregate readings into equipment states and system-level KPIs."""
        equipment_states = {}

        for unit_id, unit in self.equipment_units.items():
            unit_sensors = [
                sensor for sensor in sensor_data
                if sensor.equipment_id == unit_id
            ]
            productivity_readings = [
                sensor.reading_value for sensor in unit_sensors
                if sensor.sensor_type == 'productivity'
            ]
            energy_readings = [
                sensor.reading_value for sensor in unit_sensors
                if sensor.sensor_type == 'energy'
            ]
            maintenance_readings = [
                sensor.reading_value for sensor in unit_sensors
                if sensor.sensor_type == 'maintenance'
            ]

            equipment_states[unit_id] = {
                'current_productivity': (
                    float(np.mean(productivity_readings))
                    if productivity_readings else unit['productivity']
                ),
                'current_energy': (
                    float(np.mean(energy_readings)) if energy_readings else 0.0
                ),
                'maintenance_need': (
                    float(np.mean(maintenance_readings))
                    if maintenance_readings else 0.0
                ),
                'utilization': unit['utilization'],
                'reliability': unit['reliability'],
                'status': unit['status'],
            }

        environmental_conditions = {
            'temperature_c': float(np.random.uniform(15, 35)),
            'humidity_percent': float(np.random.uniform(40, 80)),
            'wind_speed_m_per_s': float(np.random.uniform(0, 15)),
            'visibility_fraction': float(np.random.uniform(0.5, 1.0)),
        }

        total_productivity = sum(
            state['current_productivity']
            for state in equipment_states.values()
        )
        total_energy_rate = sum(
            state['current_energy'] for state in equipment_states.values()
        )
        average_utilization = (
            float(np.mean([
                state['utilization'] for state in equipment_states.values()
            ]))
            if equipment_states else 0.0
        )
        active_units = sum(
            state['status'] == 'ACTIVE'
            for state in equipment_states.values()
        )

        operational_kpis = {
            'total_productivity': total_productivity,
            'total_energy_rate_kwh_per_hour': total_energy_rate,
            'average_utilization': average_utilization,
            'active_equipment_count': active_units,
            'system_efficiency_containers_per_kwh': (
                total_productivity / total_energy_rate
                if total_energy_rate > 0 else 0.0
            ),
            'maintenance_alerts': sum(
                state['maintenance_need'] > 0.7
                for state in equipment_states.values()
            ),
        }

        prediction_confidence = float(np.mean([
            model['accuracy'] for model in self.prediction_models.values()
        ]))
        twin_state = DigitalTwinState(
            timestamp=current_time,
            equipment_states=equipment_states,
            environmental_conditions=environmental_conditions,
            operational_kpis=operational_kpis,
            prediction_confidence=prediction_confidence,
        )
        self.digital_twin_states.append(twin_state)
        return twin_state

    def generate_predictions(
        self, horizon_hours: int = 24
    ) -> Dict[str, List[float]]:
        """Generate illustrative heuristic projections from the latest twin state."""
        if horizon_hours < 0:
            raise ValueError('The forecast horizon must be non-negative.')
        if not self.digital_twin_states:
            return {}

        latest_state = self.digital_twin_states[-1]
        predictions = {}

        for model_name, model_info in self.prediction_models.items():
            model_predictions = []
            for hour in range(horizon_hours):
                time_factor = 1.0 + hour * 0.01
                seasonal_factor = 0.9 + 0.2 * np.sin(hour * np.pi / 12)

                if model_name == 'productivity':
                    base_value = latest_state.operational_kpis['total_productivity']
                    prediction = base_value * time_factor * seasonal_factor
                elif model_name == 'energy':
                    base_value = latest_state.operational_kpis[
                        'total_energy_rate_kwh_per_hour'
                    ]
                    prediction = base_value * time_factor * seasonal_factor
                elif model_name == 'maintenance':
                    base_value = latest_state.operational_kpis['maintenance_alerts']
                    prediction = base_value * (1 + hour * 0.02)
                else:
                    base_value = 0.05
                    prediction = base_value * (1 + hour * 0.001)

                uncertainty = (1 - model_info['accuracy']) * 0.1
                noise = np.random.normal(0, uncertainty)
                model_predictions.append(max(0.0, prediction * (1 + noise)))

            predictions[model_name] = model_predictions

        return predictions

    def run_scenario_analysis(self, scenario_config: Dict) -> Dict[str, float]:
        """Evaluate a simplified scenario using fleet-level multipliers."""
        scenario_name = scenario_config.get('name', 'Unknown Scenario')
        scenario_type = scenario_config.get('type', 'standard')

        scenario_factors = {
            'standard': (1.0, 1.0, 1.0),
            'peak_demand': (1.4, 0.9, 1.2),
            'maintenance_day': (0.7, 1.0, 0.5),
            'weather_event': (0.8, 0.6, 1.1),
        }
        demand_multiplier, weather_impact, equipment_stress = (
            scenario_factors.get(scenario_type, (1.0, 1.0, 1.0))
        )

        # The failure case uses the fleet's mean reliability as expected availability.
        # It is an aggregate expectation, not a simulation of named failed units.
        availability_factor = 1.0
        if scenario_type == 'equipment_failure' and self.equipment_units:
            availability_factor = float(np.mean([
                unit['reliability'] for unit in self.equipment_units.values()
            ]))

        base_capacity = sum(
            unit['productivity'] * unit['utilization']
            for unit in self.equipment_units.values()
        )
        base_energy_rate = sum(
            self.equipment_types[unit['equipment_type']].energy_consumption
            * unit['productivity']
            * unit['utilization']
            for unit in self.equipment_units.values()
        )

        scenario_utilization = min(1.0, 0.75 * demand_multiplier)
        utilization_adjustment = scenario_utilization / 0.75
        scenario_productivity = (
            base_capacity
            * utilization_adjustment
            * weather_impact
            * availability_factor
        )
        scenario_energy_rate = (
            base_energy_rate
            * utilization_adjustment
            * equipment_stress
            * availability_factor
        )
        scenario_efficiency = (
            scenario_productivity / scenario_energy_rate
            if scenario_energy_rate > 0 else 0.0
        )

        results = {
            'productivity': scenario_productivity,
            'energy_consumption': scenario_energy_rate,
            'efficiency': scenario_efficiency,
            'utilization': scenario_utilization,
            'downtime_risk': max(0.0, equipment_stress - 1.0) * 0.1,
            'maintenance_need': equipment_stress * 1.5,
        }
        self.scenario_results[scenario_name] = results
        return results


print('Integrated Digital Twin System class defined.')

### Initialize the Digital Twin System

The fleet configuration determines how many simulated units of each type are created. Each unit receives a small random performance variation around its equipment-type specification. The fixed random seeds make the random-number sequence reproducible when the notebook is run from the beginning in the same environment; timestamps still use the runtime clock.

In [ ]:
digital_twin = IntegratedDigitalTwinSystem(equipment_types)

print('\n=== SIMULATED DIGITAL TWIN CONFIGURATION ===')
print(f'Equipment units: {len(digital_twin.equipment_units)}')
print(f'Simulated sensors: {len(digital_twin.iot_sensors)}')
print(f'Forecast categories: {len(digital_twin.prediction_models)}')
print(f'Assumed system uptime: {digital_twin.system_uptime:.1%}')
print(f'Assumed synchronization accuracy: {digital_twin.synchronization_accuracy:.1%}')
print(f'Assumed forecast accuracy: {digital_twin.prediction_accuracy:.1%}')

equipment_summary = {}
for unit in digital_twin.equipment_units.values():
    equipment_code = unit['equipment_type']
    summary = equipment_summary.setdefault(
        equipment_code,
        {'count': 0, 'productivity_sum': 0.0, 'reliability_sum': 0.0},
    )
    summary['count'] += 1
    summary['productivity_sum'] += unit['productivity']
    summary['reliability_sum'] += unit['reliability']

print('\nSimulated fleet summary:')
for equipment_code, summary in equipment_summary.items():
    count = summary['count']
    average_productivity = summary['productivity_sum'] / count
    average_reliability = summary['reliability_sum'] / count
    print(
        f'  {equipment_code}: {count} units, '
        f'mean unit capacity {average_productivity:.1f} containers/hour, '
        f'mean reliability {average_reliability:.2f}'
    )

### Run the Simulated Real-Time Digital Twin

At each simulated hour, the notebook generates a reading for every sensor, aggregates readings into a twin state, and records the state KPIs. This loop is the synchronization pattern being illustrated: observations are collected first, then the virtual representation is updated at the same simulation timestamp.

For each time step, the system efficiency is throughput divided by energy rate. Since containers per hour divided by kWh per hour gives containers per kWh, the resulting unit is consistent.

In [ ]:
print('=== SIMULATED REAL-TIME DIGITAL TWIN RUN ===')

simulation_hours = 24
current_time = datetime.now().replace(microsecond=0, second=0, minute=0)
time_series_data = {
    'timestamps': [],
    'total_productivity': [],
    'total_energy_rate': [],
    'average_utilization': [],
    'active_units': [],
    'maintenance_alerts': [],
    'system_efficiency': [],
    'prediction_confidence': [],
}
total_sensor_readings = 0

for hour in range(simulation_hours):
    simulation_time = current_time + timedelta(hours=hour)
    sensor_data = digital_twin.collect_sensor_data(simulation_time)
    total_sensor_readings += len(sensor_data)
    twin_state = digital_twin.update_digital_twin_state(
        sensor_data, simulation_time
    )
    kpis = twin_state.operational_kpis

    time_series_data['timestamps'].append(simulation_time)
    time_series_data['total_productivity'].append(kpis['total_productivity'])
    time_series_data['total_energy_rate'].append(
        kpis['total_energy_rate_kwh_per_hour']
    )
    time_series_data['average_utilization'].append(
        kpis['average_utilization']
    )
    time_series_data['active_units'].append(kpis['active_equipment_count'])
    time_series_data['maintenance_alerts'].append(kpis['maintenance_alerts'])
    time_series_data['system_efficiency'].append(
        kpis['system_efficiency_containers_per_kwh']
    )
    time_series_data['prediction_confidence'].append(
        twin_state.prediction_confidence
    )

    if (hour + 1) % 6 == 0:
        print(
            f'Hour {hour + 1}: '
            f'{kpis["total_productivity"]:.0f} containers/hour, '
            f'{kpis["system_efficiency_containers_per_kwh"]:.2f} containers/kWh, '
            f'{kpis["maintenance_alerts"]} maintenance alerts'
        )

avg_productivity = float(np.mean(time_series_data['total_productivity']))
avg_efficiency = float(np.mean(time_series_data['system_efficiency']))
avg_utilization = float(np.mean(time_series_data['average_utilization']))
max_maintenance_alerts = max(time_series_data['maintenance_alerts'])

assert len(time_series_data['timestamps']) == simulation_hours
assert len(digital_twin.digital_twin_states) == simulation_hours
assert total_sensor_readings == len(digital_twin.iot_sensors) * simulation_hours

print(f'\nSimulation duration: {simulation_hours} hours')
print(f'Twin states recorded: {len(digital_twin.digital_twin_states)}')
print(f'Simulated sensor readings: {total_sensor_readings:,}')
print('\n=== SIMULATION SUMMARY ===')
print(f'Mean productivity: {avg_productivity:.1f} containers/hour')
print(f'Mean efficiency: {avg_efficiency:.2f} containers/kWh')
print(f'Mean utilization: {avg_utilization:.1%}')
print(f'Maximum maintenance alerts in one time step: {max_maintenance_alerts}')
print('These values describe the simulation, not measured terminal performance.')

### Generate Heuristic Predictive Analytics

A forecast step takes the latest twin state and applies a simple rule over the requested horizon. For example, the productivity projection combines the latest system productivity with a gradual time factor and a periodic factor. A small random term represents modeled uncertainty. These equations are demonstrations of data flow, not learned or validated forecasting equations.

The accuracy fields are assumed metadata. They influence the size of the synthetic noise in this example, but no historical observations are available to estimate accuracy or test forecast errors.

In [ ]:
print('=== HEURISTIC FORECASTS ===')
forecast_horizon_hours = 48
predictions = digital_twin.generate_predictions(
    horizon_hours=forecast_horizon_hours
)

if not predictions:
    raise RuntimeError('A twin state is required before generating forecasts.')

for model_name, prediction_values in predictions.items():
    model_info = digital_twin.prediction_models[model_name]
    print(f'\n{model_name.title()} projection:')
    print(f'  Rule category: {model_info["model_type"]}')
    print(f'  Assumed accuracy metadata: {model_info["accuracy"]:.1%}')
    print(f'  Nominal update frequency: {model_info["update_frequency"]}')
    print(f'  Mean projected value: {np.mean(prediction_values):.2f}')
    print(
        f'  Projected range: {min(prediction_values):.2f} '
        f'to {max(prediction_values):.2f}'
    )
    if len(prediction_values) > 1:
        direction = (
            'increasing'
            if prediction_values[-1] > prediction_values[0]
            else 'non-increasing'
        )
        print(f'  Endpoint comparison: {direction}')

overall_assumed_accuracy = float(np.mean([
    model['accuracy'] for model in digital_twin.prediction_models.values()
]))
print('\n=== MODEL METADATA (NOT VALIDATION RESULTS) ===')
print(f'Mean assumed accuracy: {overall_assumed_accuracy:.1%}')
print(
    'Latest-state confidence field: '
    f'{digital_twin.digital_twin_states[-1].prediction_confidence:.1%}'
)

maintenance_predictions = predictions.get('maintenance', [])
if maintenance_predictions:
    peak_maintenance = max(maintenance_predictions)
    peak_hour = maintenance_predictions.index(peak_maintenance)
    print(
        f'\nLargest projected maintenance indicator: '
        f'hour {peak_hour + 1} ({peak_maintenance:.2f})'
    )

failure_predictions = predictions.get('failure', [])
if failure_predictions:
    print(
        'Largest heuristic failure-risk indicator: '
        f'{max(failure_predictions):.3f}'
    )

print(
    '\nInterpret these sequences as outputs of the stated rules; '
    'they are not validated operational forecasts.'
)

### Scenario Analysis and What-If Testing

Scenario analysis applies predefined multipliers to a common simulated fleet baseline. The scenario type selects demand, weather, and equipment-stress factors. Utilization is capped at 100%; throughput and energy rate are adjusted consistently, and efficiency is calculated as throughput divided by energy rate.

The equipment-failure case uses mean unit reliability as an aggregate expected-availability factor. It does not choose specific failed units or simulate failure timing. Thus, it is a coarse sensitivity case rather than a detailed reliability model. Scenario comparisons reveal the consequences of the assumptions encoded here; they do not prove that a scenario will have the same effect in a real terminal.

In [ ]:
scenarios = [
    {
        'name': 'Normal Operations',
        'type': 'standard',
        'description': 'Reference operating conditions',
    },
    {
        'name': 'Peak Demand Surge',
        'type': 'peak_demand',
        'description': 'Demand multiplier specified by the scenario rule',
    },
    {
        'name': 'Scheduled Maintenance',
        'type': 'maintenance_day',
        'description': 'Reduced demand and equipment stress in the scenario rule',
    },
    {
        'name': 'Severe Weather Event',
        'type': 'weather_event',
        'description': 'Weather impact specified by the scenario rule',
    },
    {
        'name': 'Equipment Failure',
        'type': 'equipment_failure',
        'description': 'Aggregate expected availability weighted by reliability',
    },
]

print('=== SCENARIO ANALYSIS ===')
scenario_results = {}
baseline_results = None

for scenario in scenarios:
    print(f'\n--- {scenario["name"]} ---')
    print(f'Interpretation: {scenario["description"]}')
    results = digital_twin.run_scenario_analysis(scenario)
    scenario_results[scenario['name']] = results

    if scenario['name'] == 'Normal Operations':
        baseline_results = results

    print(f'  Throughput: {results["productivity"]:.1f} containers/hour')
    print(
        '  Energy rate: '
        f'{results["energy_consumption"]:.1f} kWh/hour'
    )
    print(
        '  Efficiency: '
        f'{results["efficiency"]:.3f} containers/kWh'
    )
    print(f'  Utilization: {results["utilization"]:.1%}')
    print(f'  Simplified downtime-risk indicator: {results["downtime_risk"]:.1%}')
    print(f'  Scenario maintenance indicator: {results["maintenance_need"]:.2f}')

    if baseline_results is not None and scenario['name'] != 'Normal Operations':
        baseline_productivity = baseline_results['productivity']
        baseline_efficiency = baseline_results['efficiency']
        productivity_change = (
            (results['productivity'] / baseline_productivity - 1) * 100
            if baseline_productivity else 0.0
        )
        efficiency_change = (
            (results['efficiency'] / baseline_efficiency - 1) * 100
            if baseline_efficiency else 0.0
        )
        print(f'  Throughput change from reference: {productivity_change:+.1f}%')
        print(f'  Efficiency change from reference: {efficiency_change:+.1f}%')

assert baseline_results is not None
assert set(scenario_results) == {scenario['name'] for scenario in scenarios}

print('\n=== QUALITATIVE RESPONSE OPTIONS ===')
recommendations = {
    'Peak Demand Surge': [
        'Review whether available capacity can meet the modeled demand.',
        'Assess staffing and shift plans against operational constraints.',
        'Check that safety and equipment limits remain satisfied.',
    ],
    'Scheduled Maintenance': [
        'Compare maintenance timing with expected demand.',
        'Check whether planned work affects required service capacity.',
        'Use equipment-specific maintenance records for operational planning.',
    ],
    'Severe Weather Event': [
        'Apply terminal safety procedures and weather operating limits.',
        'Reassess throughput assumptions as conditions change.',
        'Use local forecasts and observed conditions for decisions.',
    ],
    'Equipment Failure': [
        'Identify affected units using actual condition and failure records.',
        'Assess available capacity and any required contingency actions.',
        'Do not interpret the aggregate reliability factor as a unit-level diagnosis.',
    ],
}

for scenario_name, scenario_recommendations in recommendations.items():
    if scenario_name in scenario_results:
        print(f'\n{scenario_name}:')
        for item_number, recommendation in enumerate(scenario_recommendations, 1):
            print(f'  {item_number}. {recommendation}')

### Comprehensive Visualization and Analysis

The plots show the simulated time series, heuristic productivity projection, and scenario comparisons. Productivity and efficiency have different units, so the scenario chart expresses each as an index relative to the normal-operations baseline, where 100 represents that metric's baseline value. This makes relative changes visible without placing unlike physical units on one axis.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Integrated Digital Twin Simulation', fontsize=16, fontweight='bold')

hours = np.arange(len(time_series_data['timestamps']))
ax1.plot(
    hours,
    time_series_data['total_productivity'],
    color='tab:blue',
    linewidth=2,
    label='Simulated throughput',
)
ax1.set_xlabel('Simulation hour')
ax1.set_ylabel('Throughput (containers/hour)')
ax1.set_title('Simulated Equipment Throughput')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2_twin = ax2.twinx()
line_efficiency = ax2.plot(
    hours,
    time_series_data['system_efficiency'],
    color='tab:green',
    linewidth=2,
    label='Efficiency',
)
line_energy = ax2_twin.plot(
    hours,
    time_series_data['total_energy_rate'],
    color='tab:red',
    linestyle='--',
    linewidth=2,
    label='Energy rate',
)
ax2.set_xlabel('Simulation hour')
ax2.set_ylabel('Efficiency (containers/kWh)', color='tab:green')
ax2_twin.set_ylabel('Energy rate (kWh/hour)', color='tab:red')
ax2.set_title('Efficiency and Energy Rate')
ax2.grid(True, alpha=0.3)
lines = line_efficiency + line_energy
ax2.legend(lines, [line.get_label() for line in lines], loc='best')

if predictions.get('productivity'):
    prediction_hours = np.arange(1, len(predictions['productivity']) + 1)
    ax3.plot(
        prediction_hours,
        predictions['productivity'],
        color='tab:blue',
        linewidth=2,
        label='Heuristic projection',
    )
    ax3.axhline(
        y=avg_productivity,
        color='tab:red',
        linestyle='--',
        alpha=0.8,
        label='Observed simulation mean',
    )
    ax3.set_xlabel('Forecast hour')
    ax3.set_ylabel('Throughput (containers/hour)')
    ax3.set_title('Heuristic Productivity Projection')
    ax3.grid(True, alpha=0.3)
    ax3.legend()

scenario_names = list(scenario_results)
baseline_productivity = baseline_results['productivity']
baseline_efficiency = baseline_results['efficiency']
productivity_index = [
    100 * scenario_results[name]['productivity'] / baseline_productivity
    if baseline_productivity else 0.0
    for name in scenario_names
]
efficiency_index = [
    100 * scenario_results[name]['efficiency'] / baseline_efficiency
    if baseline_efficiency else 0.0
    for name in scenario_names
]
x = np.arange(len(scenario_names))
width = 0.36
ax4.bar(
    x - width / 2,
    productivity_index,
    width,
    label='Throughput index',
    color='tab:blue',
    alpha=0.75,
)
ax4.bar(
    x + width / 2,
    efficiency_index,
    width,
    label='Efficiency index',
    color='tab:green',
    alpha=0.75,
)
ax4.axhline(100, color='black', linestyle=':', linewidth=1)
ax4.set_xlabel('Scenario')
ax4.set_ylabel('Index (normal operations = 100)')
ax4.set_title('Relative Scenario Metrics')
ax4.set_xticks(x)
ax4.set_xticklabels(
    [name.replace(' ', '\n') for name in scenario_names],
    rotation=30,
    ha='right',
)
ax4.grid(True, axis='y', alpha=0.3)
ax4.legend()

fig.tight_layout()
plt.show()
print('Visualization generated from simulated data and heuristic projections.')

### Digital Twin System Performance Summary

The summary brings together the simulation configuration, time-series metrics, forecast metadata, and scenario comparisons. Keep the distinction between three kinds of quantities in mind: simulated observations generated by the notebook, heuristic projections generated by explicit rules, and assumed metadata such as accuracy and uptime. Only the first two are computed from this run, and neither establishes real-world performance.

In [ ]:
print('=== INTEGRATED DIGITAL TWIN SIMULATION SUMMARY ===')
print('\nSIMULATION CONFIGURATION')
print(f'  Equipment units: {len(digital_twin.equipment_units)}')
print(f'  Simulated sensors: {len(digital_twin.iot_sensors)}')
print(f'  Forecast categories: {len(digital_twin.prediction_models)}')
print(f'  Simulation period: {simulation_hours} hours')
print(f'  Simulated sensor readings: {total_sensor_readings:,}')

print('\nSIMULATED OPERATIONS')
print(f'  Mean throughput: {avg_productivity:.1f} containers/hour')
print(f'  Mean efficiency: {avg_efficiency:.3f} containers/kWh')
print(f'  Mean utilization: {avg_utilization:.1%}')
print(f'  Maximum maintenance alerts in one time step: {max_maintenance_alerts}')
print(f'  Assumed system uptime metadata: {digital_twin.system_uptime:.1%}')
print(
    '  Assumed synchronization accuracy metadata: '
    f'{digital_twin.synchronization_accuracy:.1%}'
)

print('\nFORECAST METADATA')
print(f'  Mean assumed accuracy: {overall_assumed_accuracy:.1%}')
for model_name, model_info in digital_twin.prediction_models.items():
    print(
        f'  {model_name.title()} assumed accuracy: '
        f'{model_info["accuracy"]:.1%}'
    )

print('\nSCENARIO COMPARISON WITH NORMAL OPERATIONS')
for scenario_name, results in scenario_results.items():
    if scenario_name == 'Normal Operations':
        continue
    productivity_change = (
        (results['productivity'] / baseline_results['productivity'] - 1) * 100
        if baseline_results['productivity'] else 0.0
    )
    efficiency_change = (
        (results['efficiency'] / baseline_results['efficiency'] - 1) * 100
        if baseline_results['efficiency'] else 0.0
    )
    print(f'  {scenario_name}:')
    print(f'    Throughput change: {productivity_change:+.1f}%')
    print(f'    Efficiency change: {efficiency_change:+.1f}%')
    print(f'    Simplified downtime-risk indicator: {results["downtime_risk"]:.1%}')

print('\nINTERPRETATION')
print('  Sensor data, forecasts, and scenario results are simulated.')
print('  Accuracy and uptime values are assumed metadata, not measured validation results.')
print('  The equipment-failure case is an aggregate reliability-weighted estimate.')
print('  Operational use would require real data, validation, and safety review.')